# Modular latent operation (MLO): the random-target experiment

We edit the latent $z = \varphi(x)$ of a frozen text encoder through an operator on structured states, $\omega_Z(z) = \rho(z;\ s,\ \omega_S(s))$ with $s = \psi(z)$:

| piece | symbol | here |
|---|---|---|
| encoder | $\varphi$ | `Encoder("embeddinggemma", 256)`: the 256-wide prefix of the cached talent-motivation encodings |
| reader | $\psi(s \mid z)$ | `Reader(columns, hidden=2)`, cross-fitted over 5 folds with `cross_fit` |
| operator | $\omega_S$ | any function `states DataFrame -> target states DataFrame`, here `LinearSCM(...).do(X=k)` per unit |
| renderer | $\rho$ | a `Renderer` subclass: writes $s'$ back into $z$ |

**Design**

- **States.** $\psi$ reads $S^+ = \{X, T, D, U, M\}$ once; every $\rho$ conditions on $S = \{X, T, D, U\}$, so motivation `M` stays in the residual and is held constant in $Z$-space. Its preservation is read by the independent judge $\psi_{\mathrm{eval}}$.
- **No operator in training.** $\rho$ sees observed $(z, s)$ pairs only. The minimal edit (`EditorRenderer`) learns from random targets $s' \sim \mathrm{Uniform}(S)$, so it can map any $z$ to any $s'$ and operators are swapped freely at test time.
- **Estimation gap.** Every $\rho$ is trained once on the true $s$ and once on the out-of-fold reading $\psi(z)$, and tested with both: $\psi$ = 5 folds × 5 seeds = 25 fits, $\rho$ = 6 kinds × 2 × 5 seeds = 60 fits.
- **Wrong DAGs** are a test-time add-on (Section 7): same $\psi$ and $\rho$, another $\omega_S$.

The data name the sensitive attribute `X` (country-of-origin region; `O` in the paper). Fits and the evaluation are **cached** in `models/latent_lab/` (`latent_lab.cache.cached`): a cell reloads its result unless latent_lab's code, the cell's code or an input it reads changed.

Target latents $\varphi(x')$, the encodings of real texts with the target state, are used for evaluation only.

In [4]:
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from tqdm import tqdm

from exp.sim_talent_motivation.talent_motivation import outcome_bayes_predictions, talent_posterior_accuracy
from latent_lab import (
    ConstantRenderer, EditorRenderer, Encoder, FlowRenderer, GroupOTRenderer, IdentityRenderer, LinearSCM,
    OTEditorRenderer, OTFlowRenderer, Pipeline, Reader, StateEffectRenderer, cross_fit, evaluate, mean_predict,
    mean_proba, state_metrics,
)
from latent_lab.cache import cached

ROOT = Path.cwd().parent if Path.cwd().name == "latent_lab" else Path.cwd()  # repo root
SEEDS = range(5)

## 1. Data

The talent-motivation experiment (`exp/sim_talent_motivation/`): the talent SFM plus motivation `M`, a root and parent of `Y` only. Each evaluation unit has one target text, rendered with its state under its own $do(X = k)$ (`interventions.csv`).

`partition.csv` fixes the split of the 4,000 units:

| set | units | used for |
|---|---|---|
| **pool** | 2,800 in 5 folds of 560 | training $\psi$, $\rho$ and the judge |
| dev | 200 | unused here |
| **eval** | 1,000 | evaluation only; the only units with a target text $x'$ |

Within the pool, every fold holds 200 units of `L`, 200 of `P` and 160 of neither:

- **`L`** (1,000 units): the only true states $\psi$ sees. The $\psi$ of fold $k$ is fitted on `L` ∩ the other four folds (800 units) and reads all 560 units of fold $k$, so every pool unit gets an out-of-fold reading $\psi(z)$.
- **`P`** (1,000 units, disjoint from `L`): the only true states the judge $\psi_{\mathrm{eval}}$ sees. `P` sits in the folds only so that it, too, gets an out-of-fold reading.
- **$\rho$** trains on all 2,800 pool latents, with either their true states or their out-of-fold readings $\psi(z)$. So $\rho$ and the judge share the latents of `P`, and in the true-$s$ arm also their true states. $\psi$ never sees them.

Note: use `df["T"]`, never `df.T` (that is pandas' transpose).

In [5]:
DATA = ROOT / "data/sim_talent_motivation"
factual = pd.read_csv(DATA / "sim_data_factual.csv", index_col="id")
counterfactual = pd.read_csv(DATA / "sim_data_counterfactual.csv", index_col="id")  # target states, under each unit's do(X = k)
do_value = pd.read_csv(DATA / "interventions.csv", index_col="id")["do_value"]
partition = pd.read_csv(DATA / "partition.csv", index_col="id")

LEVELS = {"X": 4, "T": 3, "D": 3, "U": 3, "M": 3}
FULL = list(LEVELS)            # what psi reads
STATE = ["X", "T", "D", "U"]   # what rho conditions on

pool = partition.index[partition["partition"] == "pool"]
eval_ids = partition.index[partition["partition"] == "eval"]
folds = partition.loc[pool, "fold"].to_numpy()
labelled = partition.loc[pool, "L_rank"].gt(0).to_numpy()   # L
P = partition.index[partition["in_P"]]
assert not partition.loc[P, "L_rank"].gt(0).any(), "L and P must be disjoint"
pd.crosstab(partition.loc[pool, "fold"], partition.loc[pool, "L_rank"].gt(0).rename("in L"))

in L,False,True
fold,,
0,360,200
1,360,200
2,360,200
3,360,200
4,360,200


## 2. Encoder φ

`latents(experiment)` loads the cached encodings (no model download): `latents.factual(ids)` gives the source latents $\varphi(x)$, `latents.counterfactual(ids)` the target latents $\varphi(x')$ of the eval units.

On previous trials embeddinggemma with 768 dimensions had the best $\psi$-outputs, which is why we chose it. Smaller latent spaces and other models (Qwen and Nomic) were tried, too.

Here we use its 256-wide Matryoshka prefix (the first 256 coordinates of the cached 768-wide encodings, renormalized): at $d = 768$ the state flow's likelihood barely depends on $s$, so it hardly moves $z$. Set the width back to 768 to compare.

In [6]:
encoder = Encoder("embeddinggemma", 256)
latents = encoder.latents("talent_motivation")
z_pool, s_pool = latents.factual(pool), factual.loc[pool]
z_eval, s_eval = latents.factual(eval_ids), factual.loc[eval_ids]
z_cf, s_cf = latents.counterfactual(eval_ids), counterfactual.loc[eval_ids]
z_pool.shape, z_eval.shape

(torch.Size([2800, 256]), torch.Size([1000, 256]))

## 3. Reader ψ, cross-fitted

`cross_fit` fits one reader per fold on the labelled units (`L`) of the other four folds and reads the held-out fold, so every pool unit gets an **out-of-fold** reading $\psi(z)$ to train $\rho$ on. The eval units are read by the average of the five fold readers (`mean_predict`). Five seeds: 25 fits.

The judge $\psi_{\mathrm{eval}}$ has the same architecture, is fitted on `P`, and is used only to evaluate.

*Takes approx. 2 minutes*

In [7]:
# seed -> (out-of-fold pool readings, fold readers)
psi = {seed: cached(f"psi-seed{seed}", lambda: cross_fit(Reader(FULL, hidden=2, levels=LEVELS),
                                                       z_pool, s_pool, folds, labelled, seed=seed))
       for seed in tqdm(SEEDS, desc="psi (5 folds per seed)")}
s_read = {seed: mean_predict(readers, z_eval) for seed, (_, readers) in psi.items()}   # psi(z) on eval
judge = cached("judge", lambda: Reader(FULL, hidden=2, levels=LEVELS).fit(latents.factual(P), factual.loc[P], seed=0))


# psi on eval: per-variable accuracy, NLL and ECE; joint accuracy and NLL over S
psi_metrics = pd.concat(
    [state_metrics(mean_proba(readers, z_eval), s_eval, joint=STATE) for _, readers in psi.values()], axis=1
)
CARD = {c: int(factual[c].max()) + 1 for c in factual}
print(f"Bayes ceiling of acc_T given X, D, U and the proxies: "
      f"{talent_posterior_accuracy(s_eval, ['X', 'D', 'U', 'P', 'L', 'H', 'A'], CARD):.3f}")
oof = [(r[STATE].to_numpy() == s_pool[STATE].to_numpy()).all(1).mean() for r, _ in psi.values()]
print(f"joint accuracy over S of the out-of-fold pool readings rho trains on: {np.mean(oof):.3f}")
pd.DataFrame({
    "psi (mean)": psi_metrics.mean(axis=1),
    "psi (sd over seeds)": psi_metrics.std(axis=1),
    "judge": judge.metrics(z_eval, s_eval, joint=STATE),
}).round(3)

psi (5 folds per seed): 100%|██████████| 5/5 [01:50<00:00, 22.08s/it]


Bayes ceiling of acc_T given X, D, U and the proxies: 0.936
joint accuracy over S of the out-of-fold pool readings rho trains on: 0.655


,psi (mean),psi (sd over seeds),judge
acc_X,0.997,0.001,0.994
nll_X,0.019,0.001,0.022
ece_X,0.013,0.001,0.010
acc_T,0.774,0.008,0.756
nll_T,0.528,0.009,0.602
ece_T,0.045,0.006,0.084
acc_D,0.993,0.001,0.995
nll_D,0.032,0.004,0.023
ece_D,0.016,0.001,0.010
acc_U,0.892,0.012,0.910


### Robustness of ψ to the wording

Does $\psi$ read the meaning of a level or the codebook's phrases? 500 eval units were also rendered with a held-out paraphrase of the codebook ($\mathcal V_B$; `z_factual_vb.pt`) next to the training wording ($\mathcal V_A$). Per variable: accuracy on the same units in both wordings and the gap $\mathcal V_A - \mathcal V_B$ with a 95 % paired bootstrap CI over units ($\psi$: averaged over seeds). `X` keeps its wording and serves as negative control. The judge is fitted on $\mathcal V_A$ too, so it is reported alongside: a large gap for it means $\psi_{\mathrm{eval}}$ needs $\mathcal V_B$ texts as well.

In [8]:
vb = encoder.auxiliary("talent_motivation", "z_factual_vb.pt")
z_a, z_b, s_b = latents.factual(vb.ids), vb.factual(vb.ids), factual.loc[vb.ids]
boot = np.random.default_rng(0).integers(0, len(vb.ids), (2000, len(vb.ids)))   # paired: same units


def robustness(reads_a: list[pd.DataFrame], reads_b: list[pd.DataFrame]) -> pd.DataFrame:
    rows = {}
    for c in FULL:
        a = np.mean([r[c].to_numpy() == s_b[c].to_numpy() for r in reads_a], axis=0)   # per unit, over seeds
        b = np.mean([r[c].to_numpy() == s_b[c].to_numpy() for r in reads_b], axis=0)
        gaps = a[boot].mean(1) - b[boot].mean(1)
        rows[c] = {"V_A": a.mean(), "V_B": b.mean(), "gap": a.mean() - b.mean(),
                   "CI low": np.quantile(gaps, 0.025), "CI high": np.quantile(gaps, 0.975)}
    return pd.DataFrame(rows).T


pd.concat({
    "psi": robustness([mean_predict(r, z_a) for _, r in psi.values()], [mean_predict(r, z_b) for _, r in psi.values()]),
    "judge": robustness([judge.predict(z_a)], [judge.predict(z_b)]),
}, axis=1).round(3)

psi                               judge                             
     V_A    V_B    gap CI low CI high    V_A    V_B    gap CI low CI high
X  0.995  0.992  0.003 -0.007   0.013  0.992  0.992  0.000 -0.010   0.012
T  0.788  0.739  0.049  0.015   0.083  0.760  0.714  0.046  0.008   0.084
D  0.993  0.984  0.010 -0.000   0.020  0.994  0.988  0.006 -0.004   0.016
U  0.908  0.821  0.087  0.058   0.118  0.910  0.818  0.092  0.054   0.130
M  0.545  0.517  0.028 -0.001   0.056  0.634  0.566  0.068  0.020   0.116

## 4. Renderers ρ: 6 kinds × {true s, ψ(z)} × 5 seeds

- `flow`: constant-noise edit, $u = F^{-1}(z; s)$, $z' = F(u; s')$.
- `editor`: minimal edit, trained on random targets $s' \sim \mathrm{Uniform}(S)$ against a critic fitted on the same training states.
- `ot_editor`: the same minimal edit with an input-convex gradient map, $x' = \nabla_x \phi(x; s')$.
- `ot_flow`: shared-reference OT flow, $u = F_s^{-1}(z)$, $z' = F_{s'}(u)$, fitted by OT flow matching within each state.
- `state_effect`, `group_ot`: closed-form baselines (additive state effect, Gaussian OT between the groups of `X`).

`FlowRenderer` and `EditorRenderer` take under a minute each on CPU at $d = 256$; the whole loop runs for a while.

In [10]:
RHOS = {
    "flow": lambda: FlowRenderer(STATE, epochs=500, levels=LEVELS),
    "editor": lambda: EditorRenderer(STATE, epochs=500, levels=LEVELS),
    "editor_prox": lambda: EditorRenderer(STATE, epochs=500, proximity=0.5, levels=LEVELS),
    "ot_editor": lambda: OTEditorRenderer(STATE, epochs=500, levels=LEVELS),
    "ot_editor_prox": lambda: OTEditorRenderer(STATE, epochs=500, proximity=0.5, levels=LEVELS),
    "ot_flow": lambda: OTFlowRenderer(STATE, epochs=500, levels=LEVELS),
    "state_effect": lambda: StateEffectRenderer(STATE, levels=LEVELS),
    "group_ot": lambda: GroupOTRenderer(STATE, on="X", shrink=0.1, levels=LEVELS),
}
runs = [(kind, arm, seed) for seed in SEEDS for arm in ("true s", "ψ(z)") for kind in RHOS]
rho = {}   # (kind, train states, seed) -> fitted renderer
for kind, arm, seed in (bar := tqdm(runs, desc="rho")):
    bar.set_postfix(kind=kind, train=arm, seed=seed)
    states = s_pool if arm == "true s" else psi[seed][0]
    make = RHOS[kind]  # only this renderer's settings enter the cache key, not all of RHOS
    name = f"rho-{kind}-{'true' if arm == 'true s' else 'psi'}-seed{seed}"
    rho[(kind, arm, seed)] = cached(name, lambda: make().fit(z_pool, states, seed=seed))
len(rho)

rho: 100%|██████████| 80/80 [7:05:47<00:00, 319.34s/it, kind=group_ot, seed=4, train=ψ(z)]          


80

## 5. Operator $\omega_S$

$\rho$ was fitted without any operator; $\omega_S$ enters only now, between reading and rendering. Each eval unit's $do(X = k)$ on the SCM over $S$, with the numbers copied from `exp/sim_talent/talent_sfm.py` (**editing them changes only the operator, not the data**). Columns outside the SCM, such as `M`, pass through unchanged. `operator(seed)` builds a freshly seeded SCM, so every run sees the same abduction draws.

The SCM operator follows abduction (noise drawn from the range consistent with the observed state), action (set the do() values) and prediction (recompute the downstream nodes); `blocked={"D": ["X"]}` makes `D` read the observed `X`.

In [11]:
PARENTS = {"D": {"X": 0.30, "T": 0.60}, "U": {"X": 0.35, "T": 0.50}}   # node: {parent: weight}
NOISE = {"D": (-0.10, 0.50), "U": (-0.15, 0.50)}                        # node: (mu, sd)
do_eval = do_value.loc[eval_ids].to_numpy()


def operator(seed=0, parents=PARENTS, blocked=None):
    scm = LinearSCM({c: LEVELS[c] for c in STATE}, parents, NOISE, seed=seed)
    return scm.do(X=lambda s: do_eval, blocked=blocked)


print(LinearSCM({c: LEVELS[c] for c in STATE}, PARENTS, NOISE))
# sanity check: the operator on the true source states vs. the simulated target states
pd.DataFrame({
    "operator": operator()(s_eval[STATE].reset_index(drop=True))["U"].value_counts(normalize=True),
    "simulated": s_cf["U"].value_counts(normalize=True),
}).sort_index().round(3)

LinearSCM, roots ['X', 'T']
  D = clip(round(0.30·X + 0.60·T + ε), 0, 2),  ε ~ N(-0.10, 0.50)
  U = clip(round(0.35·X + 0.50·T + ε), 0, 2),  ε ~ N(-0.15, 0.50)


,operator,simulated
U,,
0,0.338,0.317
1,0.479,0.507
2,0.183,0.176


## 6. Downstream predictor π

A frozen MLP $\pi: Z \to Y$ (a one-column `Reader`), fitted on the observed pool latents and their outcome `Y`, with the epoch selected on a held-out 20 % of the pool. It never sees target data. References: $\pi$ on the real $\varphi(x')$, and the Bayes-optimal accuracy for $Y'$ given the true counterfactual state with and without `M`.

`score(z')` collects every metric of one edit, against the true target.

In [12]:
pi = cached("pi", lambda: Reader(["Y"], hidden=2, levels={"Y": CARD["Y"]}).fit(z_pool, s_pool, seed=0))


def predict(z: torch.Tensor) -> torch.Tensor:
    return pi.proba(z)["Y"].argmax(-1)


def score(z_prime: torch.Tensor) -> pd.Series:
    return evaluate(z_eval, z_prime, z_cf, judge, s_cf, joint=STATE,
                    predict=predict, outcome=s_cf["Y"], groups=s_cf["X"])


y_cf = s_cf["Y"].to_numpy()
pd.Series({
    "pi(z) vs factual Y": float((predict(z_eval).numpy() == s_eval["Y"].to_numpy()).mean()),
    "pi(phi(x')) vs Y'": float((predict(z_cf).numpy() == y_cf).mean()),
    **{f"Bayes vs Y' | {name}": float((p.argmax(1) == y_cf).mean())
       for name, p in outcome_bayes_predictions(s_cf).items()},
}).round(3)

pi(z) vs factual Y                                             0.665
pi(phi(x')) vs Y'                                              0.639
Bayes vs Y' | all parents (T, M known)                         0.783
Bayes vs Y' | state only (T known, M unknown)                  0.680
Bayes vs Y' | text content (X, D, U, M, proxies)               0.773
Bayes vs Y' | text content without M                           0.663
Bayes vs Y' | no talent or motivation information (X, D, U)    0.652
dtype: float64

## 7. Evaluation under each unit's do(X = k)

Every $\rho$ is tested with the true source states and with $\psi(z)$. Train × test gives the estimation gap: the `true s / true s` cell is the privileged upper bound, `ψ(z) / ψ(z)` is deployment.

- **Fidelity and preservation:** the judge's `joint_acc` over $S$ and `acc_*` on $z'$ against the true target states; `acc_M` and `acc_T` are preservation (neither changes under $do(X)$, and $\rho$ never sees `M`). `real_*` is the judge on $\varphi(x')$ itself: reading $z'$ *better* than a real latent points to off-manifold edits.
- **Latent:** `gap_closed` to the paired $\varphi(x')$ (1 = perfect, 0 = no better than $z$), `distance` $|z' - \varphi(x')|$ (the energy score of a point edit), `energy` distance between the sets $\{z'\}$ and $\{\varphi(x')\}$, `shift` $|z' - z|$; all in standardized coordinates, divided by $\sqrt d$.
- **Downstream:** `down_acc` of $\pi(z')$ against $Y'$, `down_agree` with $\pi(z)$, and `parity_gap`, the range over the target regions of the mean predicted $Y$.

In [13]:
def evaluate_all() -> pd.DataFrame:
    rows = []
    for (kind, arm, seed), renderer in tqdm(rho.items(), desc="evaluate", total=len(rho)):
        for test, states in {"true s": s_eval, "ψ(z)": s_read[seed]}.items():
            result = Pipeline(None, operator(seed), renderer)(z_eval, states)
            rows.append({"rho": kind, "train": arm, "test": test, "seed": seed, **score(result.z_prime)})
    for kind, renderer in {"identity": IdentityRenderer(STATE),
                           "constant": ConstantRenderer(STATE, on="X").fit(z_pool, s_pool)}.items():
        result = Pipeline(None, operator(), renderer)(z_eval, s_eval)
        rows.append({"rho": kind, "train": "-", "test": "true s", "seed": 0, **score(result.z_prime)})
    return pd.DataFrame(rows)


results = cached("results", evaluate_all)

metrics = ["joint_acc", "acc_X", "acc_D", "acc_U", "acc_T", "acc_M",
           "gap_closed", "distance", "energy", "shift", "down_acc", "down_agree", "parity_gap"]
print("references on the real phi(x'):", results.filter(like="real_").iloc[0].round(3).to_dict())
summary = results.groupby(["rho", "train", "test"])[metrics].agg(["mean", "std"])
summary.xs("mean", axis=1, level=1).round(3)

evaluate: 100%|██████████| 80/80 [00:52<00:00,  1.52it/s]


references on the real phi(x'): {'real_acc_X': 0.991, 'real_acc_T': 0.76, 'real_acc_D': 0.989, 'real_acc_U': 0.923, 'real_acc_M': 0.61, 'real_joint_acc': 0.686, 'real_down_acc': 0.639, 'real_down_agree': 0.548, 'real_parity_gap': 0.773}


joint_acc  acc_X  acc_D  acc_U  acc_T  acc_M  \
rho            train  test                                                   
constant       -      true s      0.125  0.500  0.560  0.502  0.762  0.624   
editor         true s true s      0.472  0.998  0.667  0.697  1.000  0.576   
                      ψ(z)        0.347  0.997  0.654  0.652  0.774  0.594   
               ψ(z)   true s      0.461  0.995  0.664  0.693  0.988  0.539   
                      ψ(z)        0.343  0.994  0.652  0.648  0.774  0.552   
editor_prox    true s true s      0.470  0.998  0.665  0.696  1.000  0.589   
                      ψ(z)        0.347  0.998  0.653  0.652  0.774  0.605   
               ψ(z)   true s      0.462  0.993  0.665  0.695  0.984  0.569   
                      ψ(z)        0.347  0.993  0.652  0.650  0.776  0.580   
flow           true s true s      0.230  0.732  0.655  0.630  0.751  0.631   
                      ψ(z)        0.219  0.732  0.645  0.618  0.734  0.631   
               ψ(z)   true s      0.217  0.720  0.655  0.612  0.757  0.632   
                      ψ(z)        0.207  0.720  0.646  0.602  0.739  0.632   
group_ot       true s true s      0.259  0.999  0.561  0.551  0.734  0.609   
                      ψ(z)        0.260  0.998  0.561  0.552  0.734  0.609   
               ψ(z)   true s      0.258  0.999  0.557  0.552  0.730  0.609   
                      ψ(z)        0.259  0.998  0.557  0.553  0.731  0.609   
identity       -      true s      0.002  0.004  0.564  0.482  0.756  0.638   
ot_editor      true s true s      0.472  0.993  0.667  0.699  1.000  0.515   
                      ψ(z)        0.348  0.993  0.654  0.654  0.774  0.529   
               ψ(z)   true s      0.463  0.991  0.666  0.696  0.991  0.445   
                      ψ(z)        0.343  0.992  0.653  0.651  0.772  0.458   
ot_editor_prox true s true s      0.471  0.991  0.668  0.700  0.999  0.550   
                      ψ(z)        0.347  0.990  0.654  0.655  0.774  0.562   
               ψ(z)   true s      0.465  0.991  0.665  0.697  0.994  0.495   
                      ψ(z)        0.345  0.990  0.652  0.652  0.775  0.510   
ot_flow        true s true s      0.346  0.997  0.666  0.664  0.764  0.624   
                      ψ(z)        0.332  0.994  0.655  0.651  0.741  0.625   
               ψ(z)   true s      0.342  0.995  0.665  0.659  0.763  0.627   
                      ψ(z)        0.327  0.994  0.654  0.645  0.744  0.626   
state_effect   true s true s      0.358  0.999  0.666  0.676  0.760  0.632   
                      ψ(z)        0.339  0.997  0.654  0.660  0.738  0.630   
               ψ(z)   true s      0.350  0.999  0.666  0.671  0.756  0.631   
                      ψ(z)        0.332  0.997  0.654  0.656  0.735  0.630   

                              gap_closed  distance  energy  shift  down_acc  \
rho            train  test                                                    
constant       -      true s       0.099     0.712   0.002  0.266     0.551   
editor         true s true s       0.066     0.735   0.005  0.472     0.629   
                      ψ(z)         0.064     0.737   0.006  0.471     0.590   
               ψ(z)   true s       0.007     0.777   0.007  0.542     0.619   
                      ψ(z)         0.007     0.777   0.009  0.536     0.586   
editor_prox    true s true s       0.097     0.714   0.003  0.392     0.630   
                      ψ(z)         0.096     0.715   0.004  0.391     0.588   
               ψ(z)   true s       0.078     0.727   0.003  0.417     0.625   
                      ψ(z)         0.079     0.727   0.004  0.413     0.590   
flow           true s true s       0.168     0.659   0.002  0.312     0.609   
                      ψ(z)         0.168     0.659   0.002  0.313     0.605   
               ψ(z)   true s       0.168     0.659   0.002  0.309     0.606   
                      ψ(z)         0.167     0.660   0.002  0.310     0.603   
group_ot       true s true s       0

In [14]:
# estimation gap per renderer: train states x test states
results[results["train"] != "-"].pivot_table(["joint_acc", "gap_closed", "down_acc"], "rho", ["train", "test"]).round(3)

down_acc                      gap_closed                       \
train            true s          ψ(z)            true s          ψ(z)          
test             true s   ψ(z) true s   ψ(z)     true s   ψ(z) true s   ψ(z)   
rho                                                                            
editor            0.629  0.590  0.619  0.586      0.066  0.064  0.007  0.007   
editor_prox       0.630  0.588  0.625  0.590      0.097  0.096  0.078  0.079   
flow              0.609  0.605  0.606  0.603      0.168  0.168  0.168  0.167   
group_ot          0.586  0.587  0.588  0.589      0.179  0.179  0.178  0.178   
ot_editor         0.603  0.575  0.604  0.573     -0.084 -0.083 -0.251 -0.249   
ot_editor_prox    0.612  0.576  0.613  0.581      0.042  0.041 -0.025 -0.024   
ot_flow           0.614  0.607  0.621  0.616      0.192  0.190  0.185  0.183   
state_effect      0.615  0.612  0.616  0.612      0.166  0.165  0.164  0.164   

               joint_acc                       
train             true s          ψ(z)         
test              true s   ψ(z) true s   ψ(z)  
rho                                            
editor             0.472  0.347  0.461  0.343  
editor_prox        0.470  0.347  0.462  0.347  
flow               0.230  0.219  0.217  0.207  
group_ot           0.259  0.260  0.258  0.259  
ot_editor          0.472  0.348  0.463  0.343  
ot_editor_prox     0.471  0.347  0.465  0.345  
ot_flow            0.346  0.332  0.342  0.327  
state_effect       0.358  0.339  0.350  0.332

## 8. Add-on: wrong DAGs at test time

$\psi$ and $\rho$ stay fixed; only $\omega_S$ changes. Shown for the flow trained and tested on $\psi(z)$, seed 0, against the true target.

In [15]:
wrong = {
    "true SCM": operator(),
    "direct effect (D, U blocked)": operator(blocked={"D": ["X"], "U": ["X"]}),
    "wrong SCM (no X->D)": operator(parents={**PARENTS, "D": {"T": 0.60}}),
}
renderer = rho[("flow", "ψ(z)", 0)]
pd.DataFrame({
    name: score(Pipeline(None, op, renderer)(z_eval, s_read[0]).z_prime)
    for name, op in wrong.items()
}).T[metrics].round(3)

,joint_acc,acc_X,acc_D,acc_U,acc_T,acc_M,gap_closed,distance,energy,shift,down_acc,down_agree,parity_gap
true SCM,0.226,0.731,0.659,0.610,0.731,0.645,0.170,0.658,0.002,0.314,0.605,0.632,0.624
"direct effect (D, U blocked)",0.155,0.602,0.563,0.522,0.744,0.639,0.151,0.672,0.002,0.255,0.543,0.850,0.096
wrong SCM (no X->D),0.183,0.682,0.564,0.601,0.757,0.631,0.160,0.665,0.002,0.278,0.559,0.768,0.285


### Add-on: M in S (held constant in S-space)

The same flow conditioned on $S^+$: the operator passes `M` through, so $\rho$ is told to keep it at $\psi$'s reading instead of leaving it in the residual. Compare `acc_M` with the flow above.

In [18]:
flow_m = cached("rho-flow-M-in-S-psi-seed0", lambda: FlowRenderer(FULL, epochs=100, levels=LEVELS).fit(z_pool, psi[0][0], seed=0))
pd.DataFrame({
    "M in residual": score(Pipeline(None, operator(), renderer)(z_eval, s_read[0]).z_prime),
    "M in S": score(Pipeline(None, operator(), flow_m)(z_eval, s_read[0]).z_prime),
}).T[metrics].round(3)

,joint_acc,acc_X,acc_D,acc_U,acc_T,acc_M,gap_closed,distance,energy,shift,down_acc,down_agree,parity_gap
M in residual,0.226,0.731,0.659,0.610,0.731,0.645,0.170,0.658,0.002,0.314,0.611,0.602,0.678
M in S,0.215,0.707,0.660,0.611,0.728,0.639,0.167,0.659,0.002,0.307,0.608,0.595,0.692
